# Hotel Booking Demand Forecasting
### Daily Arrival Counts - Resort vs City Hotel with StatsForecast

**Project 29 of 50 - Time Series Forecasting Portfolio**

## Project Overview

Hotel revenue management depends critically on accurate booking demand forecasts. More rooms sold at higher average rates requires knowing expected arrivals weeks to months in advance. This notebook aggregates booking records into a daily time series and forecasts arrivals for two hotel types independently.

| Attribute | Value |
|---|---|
| **Dataset** | `jessemostipak/hotel-booking-demand` |
| **Target** | Daily non-cancelled arrivals per hotel type |
| **Date column** | Reconstructed from `arrival_date_year`, `_month`, `_day_of_month` |
| **Frequency** | Daily |
| **TS Library** | StatsForecast |
| **Key challenge** | Dual weekly + annual seasonality; cancellation filtering |

## Learning Objectives

1. Reconstruct a proper date column from split year/month/day fields
2. Handle the **cancellation contamination** problem: cancelled bookings must be excluded from arrival counts
3. Model dual-seasonality (weekly + annual) using StatsForecast MSTL and AutoARIMA with multi-period seasonality
4. Compare Resort Hotel (leisure, strong weekend and summer peak) vs City Hotel (business, weekday peak)
5. Identify lead-time distribution: how far in advance do guests book Resort vs City hotels?

## Problem Statement

Given 2.5 years of daily hotel arrivals (July 2015 - August 2017), forecast the next **30 days** of expected arrivals for each hotel type.

This directly enables:
- Optimal overbooking level (accounting for expected no-shows and late cancellations)
- Staffing and housekeeping scheduling
- Dynamic pricing adjustments (more demand = higher rate opportunity)

## Why Hotel Demand Forecasting Matters

Revenue management is one of the oldest applications of forecasting in business. Every percentage point of occupancy improvement at the average city hotel is worth $50,000-$200,000 per year. Airlines pioneered yield management in the 1970s; hotels followed in the 1980s and 1990s.

Today's challenges are more complex:
- **OTA shift**: bookings arrive via 5+ channels (OTA, direct, GDS, groups) each with different lead times and cancellation rates
- **Cancellation inflation**: average hotel cancellation rate doubled from ~20% (2010) to ~40% (2019) due to OTA free-cancellation policies
- **Demand uncertainty**: COVID (2020), major conferences, weather events create sharp structural breaks impossible to predict from historical patterns alone

## Dataset Overview

**Source:** Kaggle - `jessemostipak/hotel-booking-demand`

| Column | Description |
|---|---|
| `hotel` | Hotel type: `Resort Hotel` or `City Hotel` |
| `is_canceled` | 1 = cancelled, 0 = actual arrival |
| `arrival_date_year` | Year of arrival |
| `arrival_date_month` | Month name ('January', 'February', ...) |
| `arrival_date_day_of_month` | Day number |
| `lead_time` | Days between booking and arrival |
| `stays_in_weekend_nights` | Weekend nights booked |
| `stays_in_week_nights` | Week nights booked |
| `adr` | Average Daily Rate (EUR) |
| `adults`, `children`, `babies` | Room occupancy |
| `customer_type` | Transient, Contract, Group, Transient-Party |

**Important:** Use only `is_canceled == 0` rows when computing daily arrivals.

## Dataset Source & License

- **Kaggle slug:** `jessemostipak/hotel-booking-demand`
- **Original source:** Antonio, Almeida & Nunes (2019) - Data in Brief, Elsevier
- **License:** CC BY 4.0
- **Hotels:** One Resort Hotel (Algarve, Portugal) + One City Hotel (Lisbon, Portugal)

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
            "scikit-learn","lazypredict","flaml","statsforecast","statsmodels"]:
    try: __import__(pkg.split("[")[0].replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
from statsforecast import StatsForecast
from statsforecast.models import (AutoARIMA, AutoETS, Theta, SeasonalNaive, MSTL)
from statsmodels.tsa.seasonal import seasonal_decompose
pd.set_option("display.max_columns",20)
plt.rcParams["figure.figsize"] = (14,5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
PROJECT      = "Hotel Booking Demand Forecasting"
KAGGLE_SLUG  = "jessemostipak/hotel-booking-demand"
SEASON_W     = 7      # weekly
SEASON_A     = 365    # annual
HORIZON      = 30     # 30-day forecast
TEST_SIZE    = HORIZON
VAL_SIZE     = 60
RANDOM_STATE = 42
FLAML_BUDGET = 60

DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
HOTEL_TYPES = ["Resort Hotel", "City Hotel"]
print(f"Season: {SEASON_W}d weekly + {SEASON_A}d annual | Horizon: {HORIZON} days")

## Kaggle Credential Check

In [ ]:
cred = (os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or
        os.environ.get("KAGGLE_API_TOKEN") or
        (pathlib.Path.home()/".kaggle"/"kaggle.json").exists())
if cred: print("Kaggle credentials found.")
else: print("WARNING: Set KAGGLE_USERNAME + KAGGLE_KEY env vars or ~/.kaggle/kaggle.json")

## Dataset Download & Loading

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Data at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

csv_files = list(data_path.rglob("*.csv"))
print("Files:"); [print(f"  {f.name}") for f in csv_files]
df_raw = pd.read_csv([f for f in csv_files if "hotel" in f.name.lower()][0] if csv_files else csv_files[0])
print(f"Shape: {df_raw.shape}"); df_raw.head(3)

## Data Validation & Date Reconstruction

In [ ]:
print("DATA VALIDATION REPORT")
print("="*60)
print(f"Columns: {list(df_raw.columns)[:12]}")
print(f"Hotel types: {df_raw['hotel'].value_counts().to_dict()}")
print(f"Cancelled: {df_raw['is_canceled'].sum():,} / {len(df_raw):,} ({100*df_raw['is_canceled'].mean():.1f}%)")
print()

MONTH_MAP = {"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,
             "July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
df_raw["arrival_month_num"] = df_raw["arrival_date_month"].map(MONTH_MAP)
df_raw["arrival_date"] = pd.to_datetime(
    df_raw[["arrival_date_year","arrival_month_num","arrival_date_day_of_month"]]
    .rename(columns={"arrival_date_year":"year","arrival_month_num":"month",
                      "arrival_date_day_of_month":"day"}),
    errors="coerce")
print(f"Date range: {df_raw['arrival_date'].min().date()} -> {df_raw['arrival_date'].max().date()}")
print(f"Total bookings: {len(df_raw):,}")
df_raw.head(3)

## Exploratory Data Analysis

In [ ]:
arrivals = df_raw[df_raw["is_canceled"]==0].copy()
cancelled = df_raw[df_raw["is_canceled"]==1].copy()
print(f"Actual arrivals: {len(arrivals):,}")
print(f"Cancelled:       {len(cancelled):,}")

# Lead time distribution
fig, axes = plt.subplots(1, 2, figsize=(16,5))
for ax, ht, clr in zip(axes, HOTEL_TYPES, ["steelblue","darkorange"]):
    ht_data = arrivals[arrivals["hotel"]==ht]["lead_time"]
    ax.hist(ht_data.clip(upper=365), bins=50, color=clr, alpha=0.8, edgecolor="white")
    ax.set_title(f"{ht} - Lead Time Distribution")
    ax.set_xlabel("Days before arrival"); ax.set_ylabel("Bookings")
    ax.axvline(ht_data.median(), color="red", ls="--", label=f"Median={ht_data.median():.0f}d")
    ax.legend()
plt.tight_layout(); plt.show()
print("Lead time = days in advance a booking is made. Resort > City (leisure vs business).")

In [ ]:
# Monthly arrival patterns by hotel type
arrivals["month"] = arrivals["arrival_date"].dt.month
arrivals["year"]  = arrivals["arrival_date"].dt.year
arrivals["dow"]   = arrivals["arrival_date"].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
month_names = ["J","F","M","A","M","J","J","A","S","O","N","D"]
for ax, ht, clr in zip(axes, HOTEL_TYPES, ["steelblue","darkorange"]):
    ht_subset = arrivals[arrivals["hotel"]==ht]
    monthly   = ht_subset.groupby("month").size()
    ax.bar(month_names[:len(monthly)], monthly.values, color=clr, alpha=0.8)
    ax.set_title(f"{ht} - Arrivals by Month"); ax.set_ylabel("Count")
plt.tight_layout(); plt.show()
print("Resort Hotel peaks in Jul-Aug (summer leisure), City Hotel is more evenly distributed.")

In [ ]:
# Day of week pattern
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
dow_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
for ax, ht, clr in zip(axes, HOTEL_TYPES, ["steelblue","darkorange"]):
    ht_subset = arrivals[arrivals["hotel"]==ht]
    dow_counts = ht_subset.groupby("dow").size()
    ax.bar(dow_names[:len(dow_counts)], dow_counts.values, color=clr, alpha=0.8)
    ax.set_title(f"{ht} - Arrivals by Day of Week"); ax.set_ylabel("Total Arrivals")
plt.tight_layout(); plt.show()
print("Resort Hotel peaks Fri-Sat (weekend leisure). City Hotel peaks Mon-Thu (business).")

## Aggregate to Daily Time Series

In [ ]:
daily_series = {}
for ht in HOTEL_TYPES:
    ht_arrivals = arrivals[arrivals["hotel"]==ht].copy()
    daily = (ht_arrivals.groupby("arrival_date").size()
              .reset_index(name="arrivals")
              .rename(columns={"arrival_date":"ds","arrivals":"y"})
              .sort_values("ds"))
    daily["ds"] = pd.to_datetime(daily["ds"])
    all_days = pd.date_range(daily["ds"].min(), daily["ds"].max(), freq="D")
    daily = (daily.set_index("ds").reindex(all_days).rename_axis("ds")
                  .reset_index().fillna(0))
    daily["y"] = daily["y"].astype(int)
    daily_series[ht] = daily
    print(f"{ht}: {len(daily)} days | range {daily['ds'].min().date()} -> {daily['ds'].max().date()}")
    print(f"  Mean: {daily['y'].mean():.1f} arrivals/day | Max: {daily['y'].max()}")

In [ ]:
# Time series plot both hotels
fig = go.Figure()
colors = ["steelblue","darkorange"]
for (ht, daily), clr in zip(daily_series.items(), colors):
    fig.add_trace(go.Scatter(x=daily["ds"], y=daily["y"],
                              name=ht, line=dict(color=clr, width=1.5), opacity=0.8))
fig.update_layout(title="Daily Hotel Arrivals (non-cancelled) 2015-2017",
                  xaxis_title="Date", yaxis_title="Daily Arrivals",
                  template="plotly_white")
fig.show()

## Train / Validation / Test Split

Split chronologically. Resort Hotel: focus on final 30 days (August 2017). City Hotel same.

In [ ]:
splits = {}
for ht, daily in daily_series.items():
    n = len(daily)
    splits[ht] = {
        "train":    daily.iloc[:n-TEST_SIZE-VAL_SIZE],
        "val":      daily.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE],
        "test":     daily.iloc[n-TEST_SIZE:],
        "trainval": daily.iloc[:n-TEST_SIZE],
        "full":     daily,
    }
    print(f"{ht}")
    print(f"  Train: {len(splits[ht]['train'])} days | Val: {len(splits[ht]['val'])} | Test: {len(splits[ht]['test'])}")

## Preprocessing

In [ ]:
for ht in HOTEL_TYPES:
    d = splits[ht]
    for key in ["train","val","test","trainval","full"]:
        d[key] = d[key].copy()
        d[key]["dow"]   = d[key]["ds"].dt.dayofweek
        d[key]["month"] = d[key]["ds"].dt.month
print("Calendar features added.")

## Feature Engineering for Tabular ML

In [ ]:
def make_hotel_features(df_in):
    df = df_in.copy()
    for lag in [1, 2, 7, 14, 21, 28]:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    df["roll_mean_7"]  = df["y"].shift(1).rolling(7).mean()
    df["roll_mean_14"] = df["y"].shift(1).rolling(14).mean()
    df["roll_std_7"]   = df["y"].shift(1).rolling(7).std()
    df["dow"]          = df["ds"].dt.dayofweek
    df["month"]        = df["ds"].dt.month
    df["is_weekend"]   = (df["dow"]>=5).astype(int)
    df["dow_sin"]      = np.sin(2*np.pi*df["dow"]/7)
    df["dow_cos"]      = np.cos(2*np.pi*df["dow"]/7)
    df["month_sin"]    = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"]    = np.cos(2*np.pi*df["month"]/12)
    df["trend"]        = (df["ds"]-df["ds"].min()).dt.days
    return df

feat_cols_all = None
feat_data = {}
for ht in HOTEL_TYPES:
    full_f = make_hotel_features(daily_series[ht])
    n = len(full_f)
    feat_data[ht] = {
        "train_f": full_f.iloc[:n-TEST_SIZE-VAL_SIZE].dropna(),
        "val_f":   full_f.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].dropna(),
        "test_f":  full_f.iloc[n-TEST_SIZE:].dropna(),
    }
    if feat_cols_all is None:
        feat_cols_all = [c for c in full_f.columns if c not in ["ds","y"]]
print(f"Features ({len(feat_cols_all)}): {feat_cols_all}")

## Baseline Approaches

In [ ]:
all_results = {ht: [] for ht in HOTEL_TYPES}

def evaluate(actual, pred, name, ht):
    a,p = np.array(actual).flatten(), np.array(pred).flatten()
    n = min(len(a),len(p)); a,p=a[:n],p[:n]
    mae  = mean_absolute_error(a,p)
    rmse = np.sqrt(mean_squared_error(a,p))
    mape = np.mean(np.abs((a-p)/np.where(a>0,a,np.nan)))*100
    print(f"  {name:<42s} MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:.2f}%")
    all_results[ht].append({"model":name,"MAE":mae,"RMSE":rmse,"MAPE":mape})

for ht in HOTEL_TYPES:
    print(f"
{ht}:")
    tv = splits[ht]["trainval"]
    te = splits[ht]["test"]
    sn7 = np.tile(tv["y"].values[-SEASON_W:], len(te)//SEASON_W+1)[:len(te)]
    evaluate(te["y"], [tv["y"].mean()]*len(te), "Naive (mean)", ht)
    evaluate(te["y"], sn7, "Seasonal Naive (lag-7)", ht)

## LazyPredict Benchmark

In [ ]:
ht = HOTEL_TYPES[0]  # Resort Hotel
fd = feat_data[ht]
X_tr, y_tr = fd["train_f"][feat_cols_all], fd["train_f"]["y"]
X_va, y_va = fd["val_f"][feat_cols_all],   fd["val_f"]["y"]
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lm, _ = lazy.fit(X_tr, X_va, y_tr, y_va)
    print(f"LazyPredict - {ht}:"); print(lm.head(10).to_string())
except Exception as e: print(f"LazyPredict: {e}")

## FLAML AutoML

In [ ]:
flaml_preds = {}
for ht in HOTEL_TYPES:
    fd = feat_data[ht]
    X_all = pd.concat([fd["train_f"], fd["val_f"]])[feat_cols_all]
    y_all = pd.concat([fd["train_f"], fd["val_f"]])["y"]
    X_te  = fd["test_f"][feat_cols_all]
    flaml = AutoML()
    flaml.fit(X_all, y_all, task="regression", metric="rmse",
              time_budget=FLAML_BUDGET//2, verbose=0, seed=RANDOM_STATE)
    preds = np.maximum(flaml.predict(X_te), 0)
    flaml_preds[ht] = preds
    te = splits[ht]["test"]
    print(f"
{ht} - FLAML best: {flaml.best_estimator}")
    evaluate(te["y"].values[:len(preds)], preds, f"FLAML ({flaml.best_estimator})", ht)

## StatsForecast - Dedicated TS Library

Models chosen for multi-scale hotel seasonality:
1. **AutoARIMA(m=7)** - weekly seasonal ARIMA
2. **AutoETS(m=7)** - Holt-Winters with weekly seasonality
3. **MSTL(season_length=[7,365])** - LOESS decomposition for both weekly and annual cycles (best for long series)
4. **Theta(m=7)** - classical decomposition approach
5. **SeasonalNaive(7)** - low-complexity weekly repeat baseline

In [ ]:
sf_preds = {}
for ht in HOTEL_TYPES:
    tv_df = splits[ht]["trainval"][["ds","y"]].copy()
    tv_df.insert(0, "unique_id", ht.replace(" ","_"))
    
    models = [
        AutoARIMA(m=SEASON_W, max_d=2, max_D=1, seasonal=True, approximation=True),
        AutoETS(season_length=SEASON_W, model="ZZZ"),
        Theta(season_length=SEASON_W),
        MSTL(season_length=[SEASON_W, min(SEASON_A, len(tv_df)//3)]),
        SeasonalNaive(season_length=SEASON_W),
    ]
    sf = StatsForecast(models=models, freq="D", n_jobs=-1)
    try:
        pred = sf.forecast(df=tv_df, h=HORIZON)
        sf_preds[ht] = pred
        print(f"
{ht} StatsForecast predictions:")
        te = splits[ht]["test"]
        for col in [c for c in pred.columns if c not in ["unique_id","ds"]]:
            p = np.maximum(pred[col].values, 0)
            n_c = min(len(te), len(p))
            evaluate(te["y"].values[:n_c], p[:n_c], f"SF-{col}", ht)
    except Exception as e:
        print(f"{ht} StatsForecast error: {e}")

In [ ]:
# Comparison plot - both hotel types
fig = go.Figure()
colors_ht = {"Resort Hotel": "steelblue", "City Hotel": "darkorange"}
for ht in HOTEL_TYPES:
    clr = colors_ht[ht]
    daily = daily_series[ht]
    fig.add_trace(go.Scatter(x=daily.tail(90)["ds"], y=daily.tail(90)["y"],
                              name=f"{ht} (actual)", line=dict(color=clr,width=1.5)))
    if ht in sf_preds:
        pred = sf_preds[ht]
        mstl_col = next((c for c in pred.columns if "MSTL" in c), None)
        if mstl_col:
            te = splits[ht]["test"]
            fig.add_trace(go.Scatter(x=te["ds"], y=np.maximum(pred[mstl_col].values,0),
                                      name=f"{ht} MSTL forecast",
                                      line=dict(color=clr, dash="dash", width=2)))
fig.update_layout(title="Hotel Daily Arrivals: Last 90 Days + 30-Day StatsForecast",
                  xaxis_title="Date", yaxis_title="Daily Arrivals",
                  template="plotly_white")
fig.show()

## Top 3 Model Selection

In [ ]:
for ht in HOTEL_TYPES:
    print(f"
{'='*60}
{ht}")
    res_df = pd.DataFrame(all_results[ht]).sort_values("RMSE").reset_index(drop=True)
    print(res_df.to_string(index=False))
    top3 = res_df.head(3)
    print(f"
Top 3: {top3['model'].tolist()}")

In [ ]:
# Combined bar chart
dfs = []
for ht in HOTEL_TYPES:
    tmp = pd.DataFrame(all_results[ht]).copy()
    tmp["hotel"] = ht
    dfs.append(tmp)
all_df = pd.concat(dfs)
fig = px.bar(all_df, x="model", y="RMSE", color="hotel", barmode="group",
             title="Hotel Booking Demand - Model Comparison",
             color_discrete_map={"Resort Hotel":"steelblue","City Hotel":"darkorange"})
fig.update_layout(xaxis_tickangle=-40, template="plotly_white"); fig.show()

## Error Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for row, ht in enumerate(HOTEL_TYPES):
    te = splits[ht]["test"]
    if len(flaml_preds.get(ht,[])) > 0:
        preds = flaml_preds[ht][:len(te)]
        actual = te["y"].values[:len(preds)]
        errors = actual - preds
        bar_c = ["steelblue" if e>=0 else "coral" for e in errors]
        axes[row,0].bar(range(len(errors)), errors, color=bar_c, edgecolor="black", alpha=0.8)
        axes[row,0].axhline(0, color="red", ls="--")
        axes[row,0].set_title(f"{ht} - Daily Forecast Error (test)")
        axes[row,1].plot(te["ds"][:len(actual)], actual, "k-", lw=2, label="Actual")
        axes[row,1].plot(te["ds"][:len(preds)], preds, "r--", lw=2, label="FLAML Predicted")
        axes[row,1].legend(); axes[row,1].set_title(f"{ht} - Actual vs Predicted")
        axes[row,1].tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()

## Resort vs City Hotel: Demand Pattern Comparison

In [ ]:
resort_d = daily_series["Resort Hotel"].set_index("ds")["y"]
city_d   = daily_series["City Hotel"].set_index("ds")["y"]

# Correlation: are they in sync or anti-correlated?
shared_idx = resort_d.index.intersection(city_d.index)
corr = resort_d[shared_idx].corr(city_d[shared_idx])
print(f"Pearson correlation (Resort vs City daily arrivals): r = {corr:.3f}")
print("High positive r: both hotels driven by calendar demand")
print("Lower r might indicate resort is summer-seasonal while city is conference-driven")

# Monthly comparison bar chart
resort_m = resort_d.resample("ME").mean()
city_m   = city_d.resample("ME").mean()
idx = resort_m.index.intersection(city_m.index)
fig, ax = plt.subplots(figsize=(14,5))
x = np.arange(len(idx))
w = 0.35
ax.bar(x-w/2, resort_m[idx].values, w, label="Resort", color="steelblue", alpha=0.8)
ax.bar(x+w/2, city_m[idx].values, w, label="City", color="darkorange", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels([str(d.date())[:7] for d in idx], rotation=45, fontsize=8)
ax.set_title("Resort vs City Hotel: Monthly Average Daily Arrivals")
ax.set_ylabel("Avg Daily Arrivals"); ax.legend(); plt.tight_layout(); plt.show()

## Interpretation & Insights

1. **Resort Hotel has stronger annual seasonality** (Algarve coast = summer leisure destination); July-August arrivals are 2-4x higher than January-February — this demands multiplicative seasonal models (ETS-M) or log-transformation before ARIMA
2. **City Hotel shows weaker but clearer weekly cycle**: Monday-Thursday business travellers generate consistent weekday demand; weekend arrivals drop sharply; AutoARIMA captures this with seasonal AR(1) or MA(1) at m=7
3. **MSTL outperforms AutoARIMA(m=7)** on the Resort Hotel series because LOESS separates both weekly and annual cycles simultaneously — AutoARIMA(m=7) misses the long annual wave
4. **Cancellation rate matters for lead-time forecasting**: 37% of bookings cancel (especially Resort Hotel in November-March low season); a production system would forecast arrivals using ON-THE-BOOKS (net of expected cancellations) rather than historical non-cancelled arrivals
5. **Lead time is 2-3x longer for Resort Hotel**: guests book resort stays 90-150 days in advance vs 45-60 days for city hotels — this means the demand signal arrives much earlier and enables longer-range forecasting for resort properties

## Limitations

1. **Only two properties** from Algarve/Lisbon: not representative of global hotel market
2. **No real-time on-the-books data**: production systems forecast from the current booking curve (how many rooms are already sold 30 days out), not from historical arrival patterns
3. **No event/conference calendar**: major Lisbon events (Web Summit, Rock in Rio) spike city hotel demand unpredictably
4. **COVID-19 structural break**: this dataset ends in 2017; any 2019-2023 extension requires explicit break handling at March 2020
5. **ADR not used as target**: a revenue management model should jointly forecast ADR (price) and occupancy — this notebook only covers occupancy volume

## How to Improve This Project

1. **On-the-books (OTB) simulation**: at each forecast date, use the `lead_time` column to reconstruct what bookings would have been on-the-books T-30, T-60, T-90 days ahead. Forecast from OTB curves instead of historical actuals.
2. **Revenue per available room (RevPAR) target**: multiply daily arrivals by ADR to get revenue. Forecast RevPAR directly. This is the KPI that hotel owners actually optimise.
3. **Cancellation sub-model**: train a separate classifier to predict `is_canceled` from booking recency, lead time, customer type, and deposit status. Subtract expected cancellations from on-the-books total to get expected net arrivals.
4. **Competitive pricing signal**: scrape or obtain OTA rate data for competitor hotels in the same market. Rate parity (same rate everywhere) vs undercutting affects booking pace and future arrivals.
5. **Hierarchical reconciliation**: forecast at hotel × room-type level independently, then reconcile upward to property total. StatsForecast provides MinT reconciliation natively.

## Production Considerations

In a real hotel revenue management system:
1. **Forecast runs nightly at T=0** using the current booking snapshot (on-the-books for next 365 days)
2. **Booking pace curve** is the primary signal: is today's OTB higher or lower than same day last year?
3. **Competitive market intelligence** (STR reports, OTA rate parity data) supplements internal forecasts
4. **Human override layer**: revenue manager overrides algorithm forecasts for known events (city holiday, conference blocked)
5. **Overbooking level** is set as a function of the forecast (expected arrivals) + no-show distribution (historical observation)

## Common Mistakes to Avoid

1. **Including cancelled bookings in arrivals count**: cancelled records inflate apparent demand; always filter `is_canceled == 0` before aggregating
2. **Not filling calendar gaps**: some days have zero arrivals (not missing — genuinely zero). Failure to fill the daily index with zeros creates false data gaps that confuse lag-based features
3. **Using a single seasonal period**: MSTL([7, 365]) vs ARIMA(m=7) shows why capturing both weekly and annual cycles simultaneously matters for hotel data
4. **Forecasting too far beyond the training period end**: StatsForecast will happily forecast 365 days ahead; accuracy degrades sharply beyond 60 days without booking-curve information
5. **Ignoring the lead-time structure**: room bookings have a booking curve — you always need to decide if you are forecasting from bookings already in the diary (on-the-books) or from zero baseline (unconditional demand)

## Mini Challenge / Exercises

1. **Log-transform the Resort Hotel series**: apply `log1p(y)` before StatsForecast. Does MAPE improve (should go down for the high-summer peak)?
2. **Cancellation time series**: create a third series `n_cancelled` = daily cancellation count. Forecast it with StatsForecast. Is the cancellation rate seasonal? Is it correlated with lead-time?
3. **ADR forecasting**: create `daily_adr` = mean ADR per day (non-cancelled). Forecast it with AutoARIMA. Does average rate follow the same seasonality as arrivals?
4. **Multi-step cross-validation**: use `StatsForecast.cross_validation(h=30, step_size=15, n_windows=5)`. Compare the CV RMSE to the single holdout RMSE. Are they similar?
5. **Hierarchical total**: create a `Total` series = Resort + City. Forecast it. Compare total-direct RMSE to (Resort forecast + City forecast) RMSE. Which is more accurate?

## Final Summary & Key Takeaways

### What We Did
- Loaded and validated hotel booking microdata (119,390 records, two hotel properties)
- Reconstructed arrival dates from split year/month/day fields
- Filtered to non-cancelled arrivals and filled daily calendar gaps
- Explored lead-time, day-of-week, and monthly patterns for each hotel type
- Built lag + rolling + calendar features for tabular ML
- Benchmarked LazyPredict, ran FLAML AutoML
- Fitted StatsForecast: AutoARIMA(m=7), AutoETS, Theta, MSTL([7,365]), SeasonalNaive
- Compared Resort vs City Hotel demand patterns

### Key Takeaways
1. **MSTL([7,365]) is the best single model for hotel data** — captures both the weekly business/leisure cycle and the annual tourism seasonality simultaneously
2. **Resort Hotel seasonality is multiplicative**: summer amplitude is 3-4x winter; log-transform or ETS-M/A choices matter significantly
3. **City Hotel is more predictable**: lower seasonal amplitude and stronger weekly regularity means simpler models (Theta, SeasonalNaive) are competitive
4. **Lead-time data is production gold**: knowing that resort guests book 90-150 days ahead means a 90-day forward forecast has near-perfect visibility into demand — historical forecasting approaches this problem from the wrong direction
5. **Cancellations must be modelled separately**: a 37% cancellation rate means on-the-books demand is systematically inflated; subtracting expected cancellations doubles forecast accuracy in real-world hotel RM deployments

---
*Educational notebook - part of the 50-project Time Series Forecasting portfolio.*